# Notebook 02 - Feature Engineering for PyTorch

## Feature Engineering Overview

In this notebook, we prepare the cleaned modeling dataset for PyTorch models. The output is a set of numerical arrays, encoders, a scaler, metadata, and class weights.

The main goal is to convert the flight data into a format that neural networks can use without changing the target definition created in Notebook 01.


In [1]:
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import pickle
import json

from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

## Data Loading

This section defines the project paths and points to the modeling parquet file created in the previous notebook.


In [2]:
BASE_DIR = Path.cwd().parent

GENERATED_DIR = BASE_DIR / "data" / "generated"
PROCESSED_DIR = GENERATED_DIR / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

model_base_path = GENERATED_DIR / "flights_2023_2025_model_base.parquet"

print("Model base path:", model_base_path)
print("Exists:", model_base_path.exists())

Model base path: c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\flights_2023_2025_model_base.parquet
Exists: True


We load the model base dataset and inspect its shape and schema before creating model-ready features.


In [3]:
df = pl.read_parquet(model_base_path)

print(df.shape)
df.head()

(20588154, 21)


YEAR,MONTH,DAY_OF_WEEK,flight_date,OP_UNIQUE_CARRIER,ORIGIN_AIRPORT_SEQ_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,DEST,DEST_CITY_NAME,DEST_STATE_ABR,ROUTE,CRS_DEP_TIME,CRS_DEP_HOUR,CRS_DEP_MINUTE,DEP_TIME_BLK,ARR_TIME_BLK,DISTANCE,delay_class,delay_class_name
i64,i64,i64,date,str,i64,str,str,str,str,str,str,str,i64,i64,i64,str,str,f64,i32,str
2023,7,1,2023-07-03,"""9E""",1013506,"""ABE""","""Allentown/Bethlehem/Easton, PA""","""PA""","""ATL""","""Atlanta, GA""","""GA""","""ABE_ATL""",654,6,54,"""0600-0659""","""0900-0959""",692.0,0,"""On time"""
2023,7,1,2023-07-03,"""9E""",1013506,"""ABE""","""Allentown/Bethlehem/Easton, PA""","""PA""","""ATL""","""Atlanta, GA""","""GA""","""ABE_ATL""",1420,14,20,"""1400-1459""","""1600-1659""",692.0,0,"""On time"""
2023,7,1,2023-07-03,"""9E""",1014602,"""ABY""","""Albany, GA""","""GA""","""ATL""","""Atlanta, GA""","""GA""","""ABY_ATL""",730,7,30,"""0700-0759""","""0800-0859""",145.0,0,"""On time"""
2023,7,1,2023-07-03,"""9E""",1014602,"""ABY""","""Albany, GA""","""GA""","""ATL""","""Atlanta, GA""","""GA""","""ABY_ATL""",1400,14,0,"""1400-1459""","""1500-1559""",145.0,0,"""On time"""
2023,7,1,2023-07-03,"""9E""",1018502,"""AEX""","""Alexandria, LA""","""LA""","""ATL""","""Atlanta, GA""","""GA""","""AEX_ATL""",612,6,12,"""0600-0659""","""0900-0959""",500.0,0,"""On time"""


In [4]:
df.columns

['YEAR',
 'MONTH',
 'DAY_OF_WEEK',
 'flight_date',
 'OP_UNIQUE_CARRIER',
 'ORIGIN_AIRPORT_SEQ_ID',
 'ORIGIN',
 'ORIGIN_CITY_NAME',
 'ORIGIN_STATE_ABR',
 'DEST',
 'DEST_CITY_NAME',
 'DEST_STATE_ABR',
 'ROUTE',
 'CRS_DEP_TIME',
 'CRS_DEP_HOUR',
 'CRS_DEP_MINUTE',
 'DEP_TIME_BLK',
 'ARR_TIME_BLK',
 'DISTANCE',
 'delay_class',
 'delay_class_name']

In [5]:
df.schema

Schema([('YEAR', Int64),
        ('MONTH', Int64),
        ('DAY_OF_WEEK', Int64),
        ('flight_date', Date),
        ('OP_UNIQUE_CARRIER', String),
        ('ORIGIN_AIRPORT_SEQ_ID', Int64),
        ('ORIGIN', String),
        ('ORIGIN_CITY_NAME', String),
        ('ORIGIN_STATE_ABR', String),
        ('DEST', String),
        ('DEST_CITY_NAME', String),
        ('DEST_STATE_ABR', String),
        ('ROUTE', String),
        ('CRS_DEP_TIME', Int64),
        ('CRS_DEP_HOUR', Int64),
        ('CRS_DEP_MINUTE', Int64),
        ('DEP_TIME_BLK', String),
        ('ARR_TIME_BLK', String),
        ('DISTANCE', Float64),
        ('delay_class', Int32),
        ('delay_class_name', String)])

Before changing the features, we check the target distribution again. This confirms that the 3-class target is still present after loading the saved file.


In [6]:
target_distribution = (
    df.group_by(["delay_class", "delay_class_name"])
      .agg(pl.len().alias("count"))
      .with_columns(
          ((pl.col("count") / df.height) * 100).round(4).alias("percentage")
      )
      .sort("delay_class")
)

target_distribution

delay_class,delay_class_name,count,percentage
i32,str,u32,f64
0,"""On time""",16357376,79.4504
1,"""Delay""",3298671,16.0222
2,"""Long delay""",932107,4.5274


In [7]:
target_distribution.write_csv(PROCESSED_DIR / "target_distribution_model_base.csv")

## Time-Based Feature Creation

Scheduled departure and arrival times are converted into minutes of the day. This makes the time values easier to use as numerical inputs.


In [8]:
df = df.with_columns([
    ((pl.col("CRS_DEP_HOUR") * 60) + pl.col("CRS_DEP_MINUTE")).alias("CRS_DEP_MINUTES_OF_DAY"),
])

Cyclic time features are created with sine and cosine transformations. This helps the model understand that times and days wrap around, such as late night being close to early morning.


In [9]:
df = df.with_columns([
    (2 * np.pi * pl.col("CRS_DEP_MINUTES_OF_DAY") / 1440).sin().alias("dep_time_sin"),
    (2 * np.pi * pl.col("CRS_DEP_MINUTES_OF_DAY") / 1440).cos().alias("dep_time_cos"),

    (2 * np.pi * pl.col("MONTH") / 12).sin().alias("month_sin"),
    (2 * np.pi * pl.col("MONTH") / 12).cos().alias("month_cos"),

    (2 * np.pi * pl.col("DAY_OF_WEEK") / 7).sin().alias("day_of_week_sin"),
    (2 * np.pi * pl.col("DAY_OF_WEEK") / 7).cos().alias("day_of_week_cos"),
])

In [10]:
df = df.with_columns(
    pl.when(pl.col("DAY_OF_WEEK").is_in([6, 7]))
      .then(1)
      .otherwise(0)
      .alias("is_weekend")
)

In [11]:
df.select([
    "MONTH",
    "DAY_OF_WEEK",
    "CRS_DEP_TIME",
    "CRS_DEP_MINUTES_OF_DAY",
    "dep_time_sin",
    "dep_time_cos",
    "DEP_TIME_BLK",
    "ARR_TIME_BLK",
    "is_weekend",
    "delay_class"
]).head()

MONTH,DAY_OF_WEEK,CRS_DEP_TIME,CRS_DEP_MINUTES_OF_DAY,dep_time_sin,dep_time_cos,DEP_TIME_BLK,ARR_TIME_BLK,is_weekend,delay_class
i64,i64,i64,i64,f64,f64,str,str,i32,i32
7,1,654,414,0.97237,-0.233445,"""0600-0659""","""0900-0959""",0,0
7,1,1420,860,-0.573576,-0.819152,"""1400-1459""","""1600-1659""",0,0
7,1,730,450,0.92388,-0.382683,"""0700-0759""","""0800-0859""",0,0
7,1,1400,840,-0.5,-0.866025,"""1400-1459""","""1500-1559""",0,0
7,1,612,372,0.99863,-0.052336,"""0600-0659""","""0900-0959""",0,0


## Categorical and Numerical Feature Selection

The selected categorical features describe airlines, airports, routes, and scheduled time blocks. The numerical features include distance and engineered time variables.


In [12]:
categorical_cols = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "ORIGIN_STATE_ABR",
    "DEST",
    "DEST_STATE_ABR",
    "ROUTE",
    "DEP_TIME_BLK",
    "ARR_TIME_BLK",
]

In [13]:
numeric_cols = [
    "DISTANCE",
    "CRS_DEP_MINUTES_OF_DAY",
    "dep_time_sin",
    "dep_time_cos",
    "month_sin",
    "month_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
]

In [14]:
target_col = "delay_class"

We check that all required columns exist and then remove rows with missing values in the selected model features.


In [15]:
missing_categorical = [col for col in categorical_cols if col not in df.columns]
missing_numeric = [col for col in numeric_cols if col not in df.columns]

print("Missing categorical columns:", missing_categorical)
print("Missing numeric columns:", missing_numeric)

Missing categorical columns: []
Missing numeric columns: []


In [16]:
required_cols = categorical_cols + numeric_cols + [target_col, "MONTH"]

null_check = df.select([
    pl.col(col).is_null().sum().alias(col)
    for col in required_cols
])

null_check

OP_UNIQUE_CARRIER,ORIGIN,ORIGIN_STATE_ABR,DEST,DEST_STATE_ABR,ROUTE,DEP_TIME_BLK,ARR_TIME_BLK,DISTANCE,CRS_DEP_MINUTES_OF_DAY,dep_time_sin,dep_time_cos,month_sin,month_cos,day_of_week_sin,day_of_week_cos,is_weekend,delay_class,MONTH
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
df_model = df.drop_nulls(required_cols)

print("Before:", df.shape)
print("After:", df_model.shape)
print("Removed:", df.height - df_model.height)

Before: (20588154, 29)
After: (20588154, 29)
Removed: 0


## Random Train/Validation/Test Stratified Split

The split is random and stratified by target class to maintain class distribution across splits:

- Train: 70% of the data
- Validation: 15% of the data
- Test: 15% of the data

This stratified approach ensures that each split has representative proportions of the target classes.

In [18]:
df_model_pd = df_model.to_pandas()

train_pd, temp_pd = train_test_split(
    df_model_pd,
    test_size=0.30,
    random_state=42,
    stratify=df_model_pd[target_col]
)

val_pd, test_pd = train_test_split(
    temp_pd,
    test_size=0.50,
    random_state=42,
    stratify=temp_pd[target_col]
)

print("Train:", train_pd.shape)
print("Validation:", val_pd.shape)
print("Test:", test_pd.shape)

Train: (14411707, 29)
Validation: (3088223, 29)
Test: (3088224, 29)


In [19]:
def get_split_distribution(data, split_name):
    distribution = (
        data[target_col]
        .value_counts()
        .sort_index()
        .reset_index()
    )

    distribution.columns = [target_col, "count"]
    distribution["percentage"] = (distribution["count"] / len(data) * 100).round(4)
    distribution["split"] = split_name

    return distribution

split_distribution = pd.concat([
    get_split_distribution(train_pd, "train"),
    get_split_distribution(val_pd, "validation"),
    get_split_distribution(test_pd, "test"),
], ignore_index=True)

split_distribution

,delay_class,count,percentage,split
0,0,11450162,79.4504,train
1,1,2309070,16.0222,train
2,2,652475,4.5274,train
3,0,2453607,79.4504,validation
4,1,494800,16.0222,validation
5,2,139816,4.5274,validation
6,0,2453607,79.4504,test
7,1,494801,16.0222,test
8,2,139816,4.5274,test


The split keeps the same general class imbalance across train, validation, and test. This is expected because long delays are rare in the full dataset.


In [20]:
split_distribution.to_csv(PROCESSED_DIR / "split_class_distribution.csv", index=False)

## Encoding Categorical Variables

The split data is converted to pandas so categorical values can be encoded as integer IDs. These IDs are later passed into embedding layers in the neural networks.


In [21]:
train_pd = train_pd[categorical_cols + numeric_cols + [target_col]].copy()
val_pd = val_pd[categorical_cols + numeric_cols + [target_col]].copy()
test_pd = test_pd[categorical_cols + numeric_cols + [target_col]].copy()

print(train_pd.shape)
print(val_pd.shape)
print(test_pd.shape)

(14411707, 18)
(3088223, 18)
(3088224, 18)


Encoders are fitted only on the training data. Unknown categories in validation or test are mapped to 0, which is reserved for unseen values.


In [22]:
category_encoders = {}
category_cardinalities = {}

for col in categorical_cols:
    unique_values = train_pd[col].astype(str).unique()
    
    encoder = {value: idx + 1 for idx, value in enumerate(unique_values)}
    
    category_encoders[col] = encoder
    category_cardinalities[col] = len(encoder) + 1  # +1 for unknown category
    
    train_pd[col] = train_pd[col].astype(str).map(encoder).fillna(0).astype("int64")
    val_pd[col] = val_pd[col].astype(str).map(encoder).fillna(0).astype("int64")
    test_pd[col] = test_pd[col].astype(str).map(encoder).fillna(0).astype("int64")

category_cardinalities

{'OP_UNIQUE_CARRIER': 16,
 'ORIGIN': 363,
 'ORIGIN_STATE_ABR': 53,
 'DEST': 363,
 'DEST_STATE_ABR': 53,
 'ROUTE': 7454,
 'DEP_TIME_BLK': 20,
 'ARR_TIME_BLK': 20}

In [23]:
with open(PROCESSED_DIR / "category_cardinalities.json", "w") as f:
    json.dump(category_cardinalities, f, indent=4)

with open(PROCESSED_DIR / "category_encoders.pkl", "wb") as f:
    pickle.dump(category_encoders, f)

## Scaling Numerical Variables

Numerical features are scaled with `StandardScaler`. The scaler is fitted on the training set and then applied to validation and test sets to avoid data leakage.


In [24]:
scaler = StandardScaler()

train_pd[numeric_cols] = scaler.fit_transform(train_pd[numeric_cols])
val_pd[numeric_cols] = scaler.transform(val_pd[numeric_cols])
test_pd[numeric_cols] = scaler.transform(test_pd[numeric_cols])

In [25]:
with open(PROCESSED_DIR / "numeric_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [26]:
train_pd[numeric_cols].describe()

,DISTANCE,CRS_DEP_MINUTES_OF_DAY,dep_time_sin,dep_time_cos,month_sin,month_cos,day_of_week_sin,day_of_week_cos,is_weekend
count,1.441171e+07,1.441171e+07,1.441171e+07,1.441171e+07,1.441171e+07,1.441171e+07,1.441171e+07,1.441171e+07,1.441171e+07
mean,7.012290e-17,1.563107e-17,-1.663341e-17,9.448304e-16,-3.952142e-17,-3.056796e-20,1.420621e-17,-1.112575e-17,-7.922920e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.378845e+00,-2.706819e+00,-1.196616e+00,-1.165208e+00,-1.391971e+00,-1.384707e+00,-1.385064e+00,-1.269715e+00,-6.191058e-01
25%,-7.317592e-01,-8.791097e-01,-9.855345e-01,-9.215609e-01,-1.202163e+00,-1.195445e+00,-1.111085e+00,-1.269715e+00,-6.191058e-01
50%,-2.564512e-01,-2.237069e-02,-2.259480e-01,-2.174417e-01,2.477838e-02,2.796636e-02,-1.767371e-03,-3.133717e-01,-6.191058e-01
75%,3.906348e-01,8.377280e-01,1.066122e+00,6.340284e-01,7.331532e-01,7.343031e-01,1.107550e+00,8.791687e-01,1.615233e+00
max,7.099983e+00,2.127876e+00,1.499044e+00,2.356440e+00,1.441528e+00,1.440640e+00,1.381529e+00,1.409899e+00,1.615233e+00


The scaled training features have means close to 0 and standard deviations close to 1. This helps neural network optimization behave more consistently.


The final feature matrices separate categorical IDs from scaled numerical features. This matches the input format used by the PyTorch datasets in the modeling notebooks.


In [27]:
X_train_cat = train_pd[categorical_cols].values.astype("int64")
X_val_cat = val_pd[categorical_cols].values.astype("int64")
X_test_cat = test_pd[categorical_cols].values.astype("int64")

X_train_num = train_pd[numeric_cols].values.astype("float32")
X_val_num = val_pd[numeric_cols].values.astype("float32")
X_test_num = test_pd[numeric_cols].values.astype("float32")

y_train = train_pd[target_col].values.astype("int64")
y_val = val_pd[target_col].values.astype("int64")
y_test = test_pd[target_col].values.astype("int64")

In [28]:
print("X_train_cat:", X_train_cat.shape)
print("X_train_num:", X_train_num.shape)
print("y_train:", y_train.shape)

print("X_val_cat:", X_val_cat.shape)
print("X_val_num:", X_val_num.shape)
print("y_val:", y_val.shape)

print("X_test_cat:", X_test_cat.shape)
print("X_test_num:", X_test_num.shape)
print("y_test:", y_test.shape)

X_train_cat: (14411707, 8)
X_train_num: (14411707, 9)
y_train: (14411707,)
X_val_cat: (3088223, 8)
X_val_num: (3088223, 9)
y_val: (3088223,)
X_test_cat: (3088224, 8)
X_test_num: (3088224, 9)
y_test: (3088224,)


## Class Weight Calculation

Class weights are calculated from the training target distribution. They reduce the effect of class imbalance by giving more weight to the less frequent classes.


In [29]:
classes = np.array(sorted(train_pd[target_col].unique()))

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights_dict = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights)
}

class_weights_dict

{0: 0.4195488529623715, 1: 2.0804489830682193, 2: 7.3625845179253355}

The highest weight is assigned to the Long delay class because it has the fewest examples. These weights are saved and reused during model training.


In [30]:
with open(PROCESSED_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights_dict, f, indent=4)

## Saving PyTorch-Ready Datasets

The arrays are saved as compressed `.npz` files. These files are smaller and faster to load than the full dataframe during model training.


In [31]:
np.savez_compressed(
    PROCESSED_DIR / "train_data.npz",
    X_cat=X_train_cat,
    X_num=X_train_num,
    y=y_train
)

np.savez_compressed(
    PROCESSED_DIR / "val_data.npz",
    X_cat=X_val_cat,
    X_num=X_val_num,
    y=y_val
)

np.savez_compressed(
    PROCESSED_DIR / "test_data.npz",
    X_cat=X_test_cat,
    X_num=X_test_num,
    y=y_test
)

Metadata is saved with the feature names, class names, category sizes, and split strategy. This keeps the modeling notebooks consistent with the preprocessing step.


{'categorical_cols': ['OP_UNIQUE_CARRIER',
  'ORIGIN',
  'ORIGIN_STATE_ABR',
  'DEST',
  'DEST_STATE_ABR',
  'ROUTE',
  'DEP_TIME_BLK',
  'ARR_TIME_BLK'],
 'numeric_cols': ['DISTANCE',
  'CRS_DEP_MINUTES_OF_DAY',
  'dep_time_sin',
  'dep_time_cos',
  'month_sin',
  'month_cos',
  'day_of_week_sin',
  'day_of_week_cos',
  'is_weekend'],
 'target_col': 'delay_class',
 'num_classes': 3,
 'class_names': {'0': 'On time', '1': 'Delay', '2': 'Long delay'},
 'category_cardinalities': {'OP_UNIQUE_CARRIER': 16,
  'ORIGIN': 363,
  'ORIGIN_STATE_ABR': 53,
  'DEST': 363,
  'DEST_STATE_ABR': 53,
  'ROUTE': 7454,
  'DEP_TIME_BLK': 20,
  'ARR_TIME_BLK': 20},
 'split_strategy': {'type': 'stratified random split',
  'train': '70%',
  'validation': '15%',
  'test': '15%',
  'random_state': 42,
  'stratified_by': 'delay_class'}}

In [33]:
print("Notebook 02 completed")

print("\nProcessed files:")
print(PROCESSED_DIR / "train_data.npz")
print(PROCESSED_DIR / "val_data.npz")
print(PROCESSED_DIR / "test_data.npz")
print(PROCESSED_DIR / "metadata.json")
print(PROCESSED_DIR / "category_cardinalities.json")
print(PROCESSED_DIR / "category_encoders.pkl")
print(PROCESSED_DIR / "numeric_scaler.pkl")
print(PROCESSED_DIR / "class_weights.json")

print("\nFinal shapes:")
print("Train categorical:", X_train_cat.shape)
print("Train numerical:", X_train_num.shape)
print("Train target:", y_train.shape)

print("Validation categorical:", X_val_cat.shape)
print("Validation numerical:", X_val_num.shape)
print("Validation target:", y_val.shape)

print("Test categorical:", X_test_cat.shape)
print("Test numerical:", X_test_num.shape)
print("Test target:", y_test.shape)

Notebook 02 completed

Processed files:
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\train_data.npz
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\val_data.npz
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\test_data.npz
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\metadata.json
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\category_cardinalities.json
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\category_encoders.pkl
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\numeric_scaler.pkl
c:\Users\of_de\Documents\ESCUELA\UP\8vo Semestre\Deep_learning\proyecto_vuelos\data\generated\processed\cla

## Notebook 02 Conclusion

This notebook created the PyTorch-ready train, validation, and test files. It also saved the categorical encoders, numerical scaler, metadata, and class weights needed by the model notebooks.
